# Robust NN Experiments — RunPod Execution Notebook

**Date:** 13 April 2026  
**Purpose:** Train ViT from scratch on 5 classification datasets with 9 robust loss functions  
**GPU:** A40 / A100 / H100 (auto-tuned batch sizes)  

---

## Datasets
| # | Dataset | Classes | Source | Domain |
|---|---------|---------|--------|--------|
| 1 | MNIST | 10 | HuggingFace | Handwritten digits |
| 2 | Fashion-MNIST | 10 | HuggingFace | Clothing items |
| 3 | CIFAR-10 | 10 | HuggingFace | Natural images |
| 4 | PathMNIST | 9 | MedMNIST | Histopathology |
| 5 | DermaMNIST | 7 | MedMNIST | Dermatoscopy |

## Loss Functions
CCE · MAE · GCE(q) · TruncGCE · SCE · DPD(β) · SDIV(β,λ) · TSCCE · ForwardT

## Pre-flight
1. Upload this notebook + `Runpod_13April2026_RobustNN_Experiments.py` to `/workspace/`
2. Select a RunPod template with **PyTorch 2.6+** (CUDA 12.x)
3. Click **Kernel → Restart and Run All**
4. **Before stopping pod:** run final cell to ZIP and download results

---

> ⚠️ **torch >= 2.6 is REQUIRED** due to CVE-2025-32434 (torch.load vulnerability).

## Cell 1 — Create Virtual Environment & Install Dependencies

**What this does:**
- Creates a Python virtual environment at `/workspace/venv` (if not exists)
- Installs ALL required packages with pinned minimum versions
- `torch>=2.6` is mandatory for CVE-2025-32434 fix
- `typing_extensions>=4.10` must be installed FIRST to avoid torch import errors
- Uses `--root-user-action=ignore` to suppress RunPod root-user pip warnings

**Known RunPod issues this fixes:**
- `ValueError: torch.load requires v2.6` → fixed by installing torch>=2.6
- `ImportError: cannot import 'override' from 'typing_extensions'` → fixed by upgrading first
- `ModuleNotFoundError: No module named 'medmnist'` → installed here

**After this cell:** If kernel restart is needed, the cell will tell you.

In [ ]:
import sys, subprocess, os

def run_pip(*pkgs, desc=""):
    """Install packages, suppressing RunPod root-user warning."""
    if desc:
        print(f"  {desc} ...")
    cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--root-user-action=ignore",
        *pkgs
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"    INSTALL ERROR: {result.stderr[-2000:]}")
        return False
    return True

print("="*60)
print("  STEP 1/5: Upgrading typing_extensions (must come first)")
print("="*60)
run_pip("typing_extensions>=4.10.0", desc="typing_extensions")

print("\n" + "="*60)
print("  STEP 2/5: Installing torch >= 2.6 with CUDA (CVE-2025-32434 fix)")
print("="*60)
import torch, subprocess as _sp, re as _re

def _detect_cuda_tag():
    """Detect CUDA version from nvidia-smi / nvcc, return PyTorch wheel tag."""
    try:
        nvcc = _sp.check_output(["nvcc", "--version"], text=True, stderr=_sp.DEVNULL)
        m = _re.search(r"release (\d+)\.(\d+)", nvcc)
        if m:
            maj, mn = int(m.group(1)), int(m.group(2))
            if maj == 12 and mn >= 4: return "cu124"
            if maj == 12 and mn >= 1: return "cu121"
            if maj == 11 and mn >= 8: return "cu118"
    except Exception:
        pass
    try:
        smi = _sp.check_output(["nvidia-smi"], text=True, stderr=_sp.DEVNULL)
        m = _re.search(r"CUDA Version: (\d+)\.(\d+)", smi)
        if m:
            maj, mn = int(m.group(1)), int(m.group(2))
            if maj == 12 and mn >= 4: return "cu124"
            if maj == 12 and mn >= 1: return "cu121"
            if maj == 11 and mn >= 8: return "cu118"
    except Exception:
        pass
    return "cu121"  # safe default for modern RunPod H100/A100 pods

tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
_has_cuda = torch.cuda.is_available()
_needs_reinstall = (tv < (2, 6)) or (not _has_cuda)

if not _has_cuda:
    print(f"  ⚠ torch {torch.__version__} has NO CUDA support (CPU-only build from PyPI)")
    print("  → Will reinstall with GPU/CUDA wheel ...")
elif tv < (2, 6):
    print(f"  torch {torch.__version__} is < 2.6 — UPGRADING with CUDA wheel ...")
else:
    print(f"  torch {torch.__version__} ✓  (>= 2.6, CUDA available — no reinstall needed)")

if _needs_reinstall:
    _cuda_tag = _detect_cuda_tag()
    _index_url = f"https://download.pytorch.org/whl/{_cuda_tag}"
    print(f"  Detected CUDA tag : {_cuda_tag}")
    print(f"  Wheel index URL   : {_index_url}")
    print("  This may take 3-5 minutes ...")
    _cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--root-user-action=ignore",
        "--index-url", _index_url,
        "torch>=2.6.0", "torchvision>=0.21.0",
    ]
    _r = _sp.run(_cmd, capture_output=True, text=True)
    if _r.returncode != 0:
        print(f"    INSTALL ERROR: {_r.stderr[-2000:]}")
    else:
        print("  ✓ torch + CUDA wheel installed.")
    print("  \n  ⚠️  torch was reinstalled — MUST restart the kernel now:")
    print("  Click: Kernel → Restart Kernel, then Run All Cells again.")

print("\n" + "="*60)
print("  STEP 3/5: HuggingFace ecosystem")
print("="*60)
run_pip(
    "transformers>=4.40.0",
    "datasets>=2.18.0",
    "accelerate>=0.30.0",
    "huggingface_hub>=0.22.0",
    "safetensors>=0.4.0",
    desc="HuggingFace stack"
)

print("\n" + "="*60)
print("  STEP 4/5: Vision & medical datasets")
print("="*60)
run_pip(
    "medmnist>=2.2.0",
    "Pillow>=10.0.0",
    desc="Vision packages"
)

print("\n" + "="*60)
print("  STEP 5/5: Science & plotting")
print("="*60)
run_pip(
    "scikit-learn>=1.3.0",
    "matplotlib>=3.8.0",
    "seaborn>=0.13.0",
    "pandas>=2.1.0",
    "numpy>=1.24.0",
    "tqdm>=4.66.0",
    desc="Science/plotting"
)

print("\n" + "="*60)
print("  ✓ Installation complete")
print(f"  Python: {sys.version}")
print("="*60)

## Cell 2 — GPU & Version Sanity Check

**What this does:**
- Confirms GPU is visible and reports VRAM, SMs, CUDA version
- Validates ALL key package versions meet minimums
- Detects AMP API version (torch.amp vs torch.cuda.amp)
- **BLOCKS execution** if torch < 2.6 (CVE-2025-32434)

**If GPU shows `None`:** Restart the RunPod pod and select a GPU tier.

**Minimum VRAM:**
- 16 GB (L4/A10G): can run all datasets with auto-tuned batch
- 24+ GB (A40/A100): recommended for full paper run

In [ ]:
import importlib.metadata as imeta
import torch
import sys

# ── GPU report ─────────────────────────────────────────────────────
print("="*60)
print("GPU STATUS")
print("="*60)
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    for i in range(n_gpus):
        d = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {d.name}")
        print(f"    VRAM  : {d.total_memory / 1024**3:.1f} GB")
        print(f"    SM    : {d.multi_processor_count} SMs")
    print(f"  CUDA version : {torch.version.cuda}")
    print(f"  PyTorch      : {torch.__version__}")
else:
    print("  ⚠ No GPU detected — running on CPU (very slow).")
    print("  Check RunPod pod settings.")

# ── torch >= 2.6 check ─────────────────────────────────────────────
tv = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
if tv < (2, 6):
    print("\n" + "!"*60)
    print("  CRITICAL: torch < 2.6 — torch.load is BLOCKED (CVE-2025-32434)")
    print("  Re-run Cell 1, then restart kernel and run all.")
    print("!"*60)
    raise RuntimeError("torch >= 2.6 required. Re-run Cell 1.")
else:
    print(f"\n  ✓ torch {torch.__version__} >= 2.6 — CVE-2025-32434 safe")

# ── AMP API ─────────────────────────────────────────────────────────
API = 'torch.amp (new)' if tv >= (2, 4) else 'torch.cuda.amp (legacy)'
MODE = 'ON' if torch.cuda.is_available() else 'OFF (CPU)'
print(f"  AMP API: {API}")
print(f"  AMP: {MODE}")

# ── Package versions ────────────────────────────────────────────────
print("\n" + "="*60)
print("PACKAGE VERSIONS")
print("="*60)

MINS = {
    "torch": (2,6,0), "torchvision": (0,16,0),
    "transformers": (4,40,0), "datasets": (2,18,0),
    "accelerate": (0,30,0), "scikit-learn": (1,3,0),
    "matplotlib": (3,8,0), "seaborn": (0,13,0),
    "pandas": (2,1,0), "numpy": (1,24,0),
    "tqdm": (4,66,0), "medmnist": (2,2,0),
    "typing_extensions": (4,10,0), "safetensors": (0,4,0),
}

all_ok = True
for pkg, minv in MINS.items():
    try:
        ver = imeta.version(pkg)
        parts = [int(x) for x in ver.split('+')[0].split('.')[:3]]
        while len(parts) < 3: parts.append(0)
        ok = tuple(parts) >= minv
        mark = "✓" if ok else "✗ NEEDS UPDATE"
        if not ok: all_ok = False
        print(f"  {pkg:<25} {ver:<15} {mark}")
    except imeta.PackageNotFoundError:
        print(f"  {pkg:<25} {'MISSING':<15} ✗")
        all_ok = False

print("\n" + ("✓ All packages OK" if all_ok else "✗ Some packages need updating — re-run Cell 1"))

## Cell 3 — Workspace & Environment Configuration

**What this does:**
- Sets `/workspace/` as working directory (RunPod persistent storage)
- Configures HuggingFace cache inside `/workspace/` (survives pod restarts)
- Locates the companion `.py` experiment script
- Sets experiment parameters via environment variables
- **Auto-detects GPU tier** and sets optimal batch size

**Configurable parameters:**
| Variable | Quick Run | Full Paper | What it does |
|----------|-----------|------------|------|
| `ROBUST_NN_QUICK_RUN` | `1` (default) | `0` | Quick debug vs full run |
| `ROBUST_NN_VIT_EPOCHS` | `30` | `250` | ViT training epochs |
| `ROBUST_NN_VIT_BATCH` | `0` (auto) | `0` (auto) | 0 = auto-tune by VRAM |
| `ROBUST_NN_DATASETS` | all 5 | all 5 | Comma-separated dataset list |

In [ ]:
import os, sys
from pathlib import Path

# ── Working directory ──────────────────────────────────────────────
WORKSPACE = Path("/workspace")
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)
print(f"Working directory: {os.getcwd()}")

# ── Results directory ──────────────────────────────────────────────
RESULTS_DIR = WORKSPACE / "results_13April2026"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results directory: {RESULTS_DIR}")

# ── HuggingFace cache ──────────────────────────────────────────────
HF_HOME = str(WORKSPACE / "hf_cache")
os.environ["HF_HOME"] = HF_HOME
os.environ["TRANSFORMERS_CACHE"] = HF_HOME
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / "hf_datasets_cache")
Path(HF_HOME).mkdir(parents=True, exist_ok=True)
Path(os.environ["HF_DATASETS_CACHE"]).mkdir(parents=True, exist_ok=True)
print(f"HF cache: {HF_HOME}")

# ══════════════════════════════════════════════════════════════════
#   EXPERIMENT CONFIGURATION — EDIT HERE
# ══════════════════════════════════════════════════════════════════
os.environ["ROBUST_NN_QUICK_RUN"]     = "1"       # ← "0" for full paper run
os.environ["ROBUST_NN_PART"]          = "A"#"AB"       # A=Vision, B=NLP, AB=both
os.environ["ROBUST_NN_VIT_EPOCHS"]    = "10"#"30"      # ← "250" for full paper
os.environ["ROBUST_NN_VIT_BATCH"]     = "0"       # 0 = auto by GPU VRAM
os.environ["ROBUST_NN_NUM_WORKERS"]   = "0"       # MUST be 0 in Jupyter (avoids CPU deadlock)
os.environ["ROBUST_NN_NLP_EPOCHS"]    = "3"
os.environ["ROBUST_NN_NLP_BATCH"]     = "32"
os.environ["ROBUST_NN_NLP_MAX_TRAIN"] = "1500"
os.environ["ROBUST_NN_RESULTS_DIR"]   = str(RESULTS_DIR)
os.environ["ROBUST_NN_SEED"]          = "42"

# ALL 5 DATASETS:
os.environ["ROBUST_NN_DATASETS"] = "pathmnist,dermamnist" #"mnist,fashion_mnist,cifar10,pathmnist,dermamnist"
# ══════════════════════════════════════════════════════════════════

# ── Locate experiment script ──────────────────────────────────────
def find_script(fname):
    search = [Path.cwd(), Path("/workspace"), Path("/workspace/code"),
              Path("/workspace/Robust-NN-learning/code"),
              Path("/workspace/Robust-NN-learning")]
    for base in search:
        p = base / fname
        if p.is_file(): return p.resolve()
    raise FileNotFoundError(
        f"Cannot find '{fname}'.\n"
        f"Upload it to /workspace/ (same folder as this notebook).\n"
        f"Searched: {[str(b) for b in search]}")

SCRIPT = find_script("Runpod_13April2026_RobustNN_Experiments.py")
if str(SCRIPT.parent) not in sys.path:
    sys.path.insert(0, str(SCRIPT.parent))

print(f"\nExperiment script: {SCRIPT}")
print("\nEnvironment config:")
for k, v in sorted(os.environ.items()):
    if k.startswith("ROBUST_NN"):
        print(f"  {k} = {v}")
print("\n✓ Workspace setup complete.")

## Cell 4 — Matplotlib Headless Configuration

**What this does:**
- Forces `Agg` backend (no X11 display server on RunPod)
- Sets publication-quality figure defaults
- Provides helper function to display saved plots inline

**Why `Agg`:** RunPod containers have no display. Using `TkAgg` crashes with `cannot connect to X server`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPyImage

plt.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 150,
    'font.size': 11, 'axes.titlesize': 11,
    'axes.labelsize': 10, 'legend.fontsize': 9,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
})

def show_all_plots(pattern="*.png", max_plots=50):
    plots = sorted(RESULTS_DIR.glob(pattern))
    if not plots:
        print(f"No plots found in {RESULTS_DIR}.")
        return
    print(f"Displaying {min(len(plots), max_plots)} of {len(plots)} plots:")
    for p in plots[:max_plots]:
        print(f"\n--- {p.name} ---")
        display(IPyImage(filename=str(p), width=900))

print("✓ Matplotlib configured (Agg backend)")
print(f"  Plots → {RESULTS_DIR}")

## Cell 5 — Import Verification

**What this does:**
- Tests every Python import needed before experiments run
- Reports which Parts (A=Vision, B=NLP) are available
- Catches and reports specific import errors with fix instructions

**If ImportError:** Re-run Cell 1, restart kernel, run all.

In [ ]:
import importlib

def try_import(name, friendly=None):
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, '__version__', 'ok')
        print(f"  ✓ {friendly or name:<25} {ver}")
        return True
    except ImportError as e:
        print(f"  ✗ {friendly or name:<25} MISSING — {e}")
        return False

print("Required packages:")
req = all([
    try_import("torch"), try_import("torchvision"),
    try_import("numpy"), try_import("pandas"),
    try_import("matplotlib"), try_import("seaborn"),
    try_import("sklearn", "scikit-learn"),
    try_import("tqdm"),
])

print("\nOptional packages:")
has_hf  = try_import("transformers") and try_import("datasets")
has_med = try_import("medmnist")

print("\nAvailability:")
print(f"  Part A (Vision ViT)   : {'✓' if req else '✗'}")
print(f"  Part A (PathMNIST)    : {'✓' if has_med else '✗ install medmnist'}")
print(f"  Part A (DermaMNIST)   : {'✓' if has_med else '✗ install medmnist'}")
print(f"  Part B (NLP BERT)     : {'✓' if has_hf else '✗ install transformers+datasets'}")

if not req:
    raise RuntimeError("Missing required packages. Re-run Cell 1.")
print("\n✓ All required imports OK.")

---

# Part A — Vision Experiments

Each cell below runs the full experiment battery for ONE dataset:
- Trains ViT from scratch with ALL 9 loss functions
- Tests 5 noise rates: η = {0, 0.1, 0.2, 0.3, 0.4}
- FGSM adversarial attacks: ε = {0, 1/255, 2/255, 4/255, 8/255}
- SDIV (β,λ) 3D accuracy surface
- Curriculum GCE annealing

**You can run cells individually** (one dataset at a time) or use Cell 12 to run all at once.

---

## Cell 6 — MNIST Experiments

**Dataset:** 60K train / 10K test, 10 classes (digits 0-9)
**Expected runtime (A40, 30 epochs):** ~30-45 min

**Outputs:**
- `mnist_curves_eta*.png` — per-loss training curves at each noise rate
- `mnist_confmat_*.png` — normalized confusion matrices per loss
- `mnist_robustness_noise.png` — accuracy vs η (PRIMARY figure)
- `mnist_robustness_fgsm.png` — accuracy vs ε
- `mnist_dual_frontier.png` — Pareto frontier scatter
- `mnist_summary_bar.png` — bar chart comparison
- `mnist_sdiv_surface_3d.png` — SDIV hyperparameter surface
- `mnist_curriculum_gce_eta0.3.png` — annealing schedule + accuracy

In [ ]:
# import runpy, os, torch

# os.environ["ROBUST_NN_PART"] = "A"
# os.environ["ROBUST_NN_DATASETS"] = "mnist"

# print("Starting: MNIST experiments")
# print(f"  GPU: {torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'}")
# print("="*60)

# try:
#     runpy.run_path(str(SCRIPT), run_name="__main__")
#     print("\n✓ MNIST experiments completed.")
# except Exception as e:
#     import traceback
#     print(f"\n✗ Failed: {type(e).__name__}: {e}")
#     traceback.print_exc()
#     print("\nTroubleshooting:")
#     print("  CUDA OOM     → reduce VIT_BATCH in Cell 3 (try 128)")
#     print("  torch.load   → upgrade torch>=2.6 (Cell 1)")
#     print("  HF download  → check internet; set HF_TOKEN if needed")
# finally:
#     if torch.cuda.is_available(): torch.cuda.empty_cache()

## Cell 7 — Fashion-MNIST Experiments

**Dataset:** 60K train / 10K test, 10 classes (clothing items)
**More challenging** than MNIST — robust losses should show clearer separation.

In [ ]:
# import runpy, os, torch

# os.environ["ROBUST_NN_PART"] = "A"
# os.environ["ROBUST_NN_DATASETS"] = "fashion_mnist"

# print("Starting: Fashion-MNIST experiments")
# print("="*60)

# try:
#     runpy.run_path(str(SCRIPT), run_name="__main__")
#     print("\n✓ Fashion-MNIST experiments completed.")
# except Exception as e:
#     import traceback
#     print(f"\n✗ Failed: {type(e).__name__}: {e}")
#     traceback.print_exc()
# finally:
#     if torch.cuda.is_available(): torch.cuda.empty_cache()

## Cell 8 — CIFAR-10 Experiments

**Dataset:** 50K train / 10K test, 10 classes (natural images), 3-channel RGB
**Most challenging** standard benchmark — ViT will have lower accuracy than MNIST.

In [ ]:
# import runpy, os, torch

# os.environ["ROBUST_NN_PART"] = "A"
# os.environ["ROBUST_NN_DATASETS"] = "cifar10"

# print("Starting: CIFAR-10 experiments")
# print("="*60)

# try:
#     runpy.run_path(str(SCRIPT), run_name="__main__")
#     print("\n✓ CIFAR-10 experiments completed.")
# except Exception as e:
#     import traceback
#     print(f"\n✗ Failed: {type(e).__name__}: {e}")
#     traceback.print_exc()
# finally:
#     if torch.cuda.is_available(): torch.cuda.empty_cache()

## Cell 9 — PathMNIST Experiments (Histopathology)

**Dataset:** ~90K train / ~7K test, 9 classes (colon tissue types)
**Medical imaging** — this is the dataset from https://huggingface.co/datasets/jafermarq/pathmnist
Loaded via the `medmnist` library for proper train/val/test splits.

**Why training matters here:** Previous code used CLIP zero-shot (~18% acc, random-level).
Training from scratch gives meaningful confusion matrices and robustness curves.

In [ ]:
import runpy, os, torch

os.environ["ROBUST_NN_PART"] = "A"
os.environ["ROBUST_NN_DATASETS"] = "pathmnist"

print("Starting: PathMNIST (histopathology) experiments")
print("="*60)

try:
    runpy.run_path(str(SCRIPT), run_name="__main__")
    print("\n✓ PathMNIST experiments completed.")
except Exception as e:
    import traceback
    print(f"\n✗ Failed: {type(e).__name__}: {e}")
    traceback.print_exc()
    print("\nIf medmnist error: pip install medmnist>=2.2.0")
finally:
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## Cell 10 — DermaMNIST Experiments (Dermatoscopy)

**Dataset:** ~7K train / ~2K test, 7 classes (skin lesion types)
**Medical imaging** — from https://huggingface.co/datasets/OctoMed/DermaMNIST
Loaded via `medmnist` library.

**Smaller dataset** — should train faster but may show class imbalance effects.

In [ ]:
import runpy, os, torch

os.environ["ROBUST_NN_PART"] = "A"
os.environ["ROBUST_NN_DATASETS"] = "dermamnist"

print("Starting: DermaMNIST (dermatoscopy) experiments")
print("="*60)

try:
    runpy.run_path(str(SCRIPT), run_name="__main__")
    print("\n✓ DermaMNIST experiments completed.")
except Exception as e:
    import traceback
    print(f"\n✗ Failed: {type(e).__name__}: {e}")
    traceback.print_exc()
    print("\nIf medmnist error: pip install medmnist>=2.2.0")
finally:
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## Cell 11b — Part C: Pretrained Model Fine-Tuning (Medical Datasets)

**What this does:**
- Fine-tunes 3 pretrained foundation models on PathMNIST + DermaMNIST
- Uses **linear probe**: freezes backbone, trains only classifier head
- Tests all 9 robust loss functions under 5 noise rates
- Compares: ViT-Scratch vs ViT-Base vs CLIP vs MedSigLIP

**Models:**
| Model | Params | Input | Domain |
|-------|--------|-------|--------|
| `google/vit-base-patch16-224` | 86M | 224×224 | ImageNet |
| `openai/clip-vit-base-patch32` | 86M | 224×224 | Internet images+text |
| `google/medsiglip-448` | 400M | 448×448 | Medical (pathology+derm) |

**VRAM needed:** ~4-8 GB (linear probe) per model
**Runtime (A40, 20 epochs):** ~1-2 hours for all 3 models × 2 datasets

**⚠️ MedSigLIP requires:**
1. Accept terms at https://huggingface.co/google/medsiglip-448
2. Set `HF_TOKEN` in Cell 3 (add: `os.environ['HF_TOKEN'] = 'hf_...'`)
3. If you skip MedSigLIP, ViT-Base + CLIP will still run fine

In [ ]:
import runpy, os, torch

os.environ["ROBUST_NN_PART"] = "C"
os.environ["ROBUST_NN_DATASETS"] = "pathmnist,dermamnist"

# Optional: set HF_TOKEN for MedSigLIP (gated model)
# os.environ["HF_TOKEN"] = "hf_..."  # ← paste your token here

print("Starting: Pretrained Model Fine-Tuning (Part C)")
print("  Models: ViT-Base, CLIP-ViT, MedSigLIP")
print("  Datasets: PathMNIST, DermaMNIST")
print("="*60)

try:
    runpy.run_path(str(SCRIPT), run_name="__main__")
    print("\n✓ Part C completed.")
except Exception as e:
    import traceback
    print(f"\n✗ Failed: {type(e).__name__}: {e}")
    traceback.print_exc()
    print("\nTroubleshooting:")
    print("  MedSigLIP gate → set HF_TOKEN and accept terms")
    print("  CUDA OOM       → MedSigLIP needs ~8GB VRAM")
finally:
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## Cell 11 — Part B: NLP BERT Fine-tuning

**Datasets:** Emotion (6 classes) + PubMedQA (3 classes)
**Model:** DistilBERT / SciBERT fine-tuned with each robust loss
**Uses `safetensors`** to avoid torch.load CVE

**Expected runtime (A40):** ~20-40 min

In [ ]:
# import runpy, os, torch

# os.environ["ROBUST_NN_PART"] = "B"

# print("Starting: NLP BERT fine-tuning")
# print("="*60)

# try:
#     runpy.run_path(str(SCRIPT), run_name="__main__")
#     print("\n✓ NLP experiments completed.")
# except ImportError as e:
#     print(f"\n✗ ImportError: {e}")
#     print("  Install transformers/datasets (Cell 1)")
# except Exception as e:
#     import traceback
#     print(f"\n✗ Failed: {type(e).__name__}: {e}")
#     traceback.print_exc()
# finally:
#     if torch.cuda.is_available(): torch.cuda.empty_cache()

---

# Results & Visualization

---

## Cell 12 — Run ALL Datasets at Once

**Alternative to Cells 6-11:** Runs all 5 vision datasets + NLP sequentially.
**Recommended for:** Overnight runs on A100/H100.

**Estimated time (Quick, 30 ep):** ~3-5 hours
**Estimated time (Full, 250 ep):** ~24-48 hours

In [ ]:
import runpy, os, time, torch

os.environ["ROBUST_NN_PART"] = "AB"
os.environ["ROBUST_NN_DATASETS"] = "mnist,fashion_mnist,cifar10,pathmnist,dermamnist"

print("Starting ALL experiments (5 datasets + NLP)")
print(f"  Quick: {os.environ.get('ROBUST_NN_QUICK_RUN', '1')}")
print(f"  Epochs: {os.environ.get('ROBUST_NN_VIT_EPOCHS', '30')}")
print("="*60)

t0 = time.time()
try:
    runpy.run_path(str(SCRIPT), run_name="__main__")
    elapsed = time.time() - t0
    h, rem = divmod(elapsed, 3600)
    m, s = divmod(rem, 60)
    print(f"\n✓ ALL experiments done. Time: {int(h)}h {int(m)}m {s:.0f}s")
except Exception as e:
    import traceback
    elapsed = time.time() - t0
    print(f"\n✗ Failed after {elapsed/60:.1f} min: {e}")
    traceback.print_exc()
finally:
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## Cell 13 — List All Output Files

**What this does:** Lists every CSV and PNG in the results directory with file sizes.

In [ ]:
from pathlib import Path

if not RESULTS_DIR.exists():
    print(f"No results yet: {RESULTS_DIR}")
else:
    all_files = sorted(RESULTS_DIR.iterdir())
    csvs = [f for f in all_files if f.suffix == '.csv']
    pngs = [f for f in all_files if f.suffix == '.png']
    print(f"Results: {len(all_files)} files ({len(csvs)} CSV, {len(pngs)} PNG)")
    if csvs:
        print("\n--- CSV files ---")
        for f in csvs:
            print(f"  {f.name:<55} {f.stat().st_size/1024:7.1f} KB")
    if pngs:
        print("\n--- PNG plots ---")
        for f in pngs:
            print(f"  {f.name:<55} {f.stat().st_size/1024:7.1f} KB")
    total_mb = sum(f.stat().st_size for f in all_files if f.is_file()) / 1024**2
    print(f"\nTotal: {total_mb:.1f} MB")

## Cell 14 — Display All Plots Inline

**Key plots to examine:**

| Plot | What to check |
|------|---------------|
| `*_robustness_noise.png` | Which loss degrades least as η increases |
| `*_curves_eta*.png` | Each loss has own Y-axis (not mixed!) |
| `*_confmat_*.png` | Row-normalized values, actual class names |
| `*_dual_frontier.png` | Top-right = best on BOTH noise + adversarial |
| `*_summary_bar.png` | Quick comparison across losses at key noise rates |

In [ ]:
from IPython.display import display, Image as IPyImage

if not RESULTS_DIR.exists():
    print("Run experiments first.")
else:
    GROUPS = [
        ("Robustness curves (PRIMARY)", "*robustness_noise*"),
        ("Training curves", "*curves_eta0.0*"),
        ("Training under noise (η=0.3)", "*curves_eta0.3*"),
        ("Confusion matrices (clean)", "*confmat*eta0.0*"),
        ("Confusion matrices (noisy)", "*confmat*eta0.3*"),
        ("FGSM adversarial", "*robustness_fgsm*"),
        ("Dual frontier", "*dual_frontier*"),
        ("Summary bars", "*summary_bar*"),
        ("SDIV surfaces", "*sdiv_surface*"),
        ("Curriculum annealing", "*curriculum*"),
    ]
    displayed = set()
    for group_name, pattern in GROUPS:
        files = [f for f in sorted(RESULTS_DIR.glob(pattern)) if f.suffix == '.png']
        new_files = [f for f in files if f not in displayed]
        if new_files:
            print(f"\n{'='*60}")
            print(f"  {group_name}")
            print(f"{'='*60}")
            for f in new_files:
                print(f"  {f.name}")
                display(IPyImage(filename=str(f), width=900))
                displayed.add(f)
    
    remaining = [f for f in sorted(RESULTS_DIR.glob("*.png")) if f not in displayed]
    if remaining:
        print(f"\n{'='*60}\n  Other plots\n{'='*60}")
        for f in remaining:
            print(f"  {f.name}")
            display(IPyImage(filename=str(f), width=900))

## Cell 15 — Summary Results Tables

**What this does:**
- Loads noise and FGSM CSVs for each dataset
- Pivots into loss × noise_rate accuracy table
- Highlights best loss at each noise level
- Shows which loss is most robust overall

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

for ds in ["mnist", "fashion_mnist", "cifar10", "pathmnist", "dermamnist"]:
    for suffix, title_type in [("noise_results", "Label Noise"), ("fgsm_results", "FGSM")]:
        fpath = RESULTS_DIR / f"{ds}_{suffix}.csv"
        if not fpath.exists():
            continue
        df = pd.read_csv(fpath)
        col = "noise_rate" if "noise" in suffix else "epsilon"
        agg = df.groupby(["loss", col])["accuracy"].mean().reset_index()
        pivot = agg.pivot(index="loss", columns=col, values="accuracy")
        
        print(f"\n{'='*70}")
        print(f"  {ds.upper()} — {title_type} Results")
        print(f"{'='*70}")
        print(pivot.to_string())
        print(f"\nBest loss per {col}:")
        for c in pivot.columns:
            best = pivot[c].idxmax()
            print(f"  {col}={c:.4g} → {best} (acc={pivot[c].max():.4f})")

## Cell 16 — OOM Troubleshooting

**Run ONLY if you got CUDA Out-of-Memory.**

Clears GPU cache and recommends batch sizes for your GPU tier.
Then edit Cell 3 accordingly and re-run the failed experiment cell.

In [ ]:
import gc, torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    alloc = torch.cuda.memory_allocated() / 1024**3
    print(f"GPU: {total:.1f} GB total, {alloc:.2f} GB allocated")
    if total < 10:
        print("  8GB: VIT_BATCH=64, NLP_BATCH=8")
    elif total < 20:
        print("  16GB: VIT_BATCH=128, NLP_BATCH=16")
    elif total < 28:
        print("  24GB: VIT_BATCH=256, NLP_BATCH=32")
    else:
        print(f"  {total:.0f}GB: VIT_BATCH=512+, NLP_BATCH=64")
else:
    print("No GPU. Use VIT_BATCH=32, NLP_BATCH=8")

## Cell 17 — FINAL: Archive Results for Download

**⚠️ Run this BEFORE stopping the RunPod pod!**

Creates a ZIP of all results (CSV + PNG). Download from:
RunPod Dashboard → Pod → Files → `/workspace/runpod_results_*.zip`

In [ ]:
import shutil, time
from pathlib import Path
from IPython.display import display, HTML

if not RESULTS_DIR.exists() or not any(RESULTS_DIR.iterdir()):
    print("No results to archive. Run experiments first.")
else:
    all_files = list(RESULTS_DIR.glob("**/*"))
    n_files = sum(1 for f in all_files if f.is_file())
    total_mb = sum(f.stat().st_size for f in all_files if f.is_file()) / 1024**2
    print(f"Archiving {n_files} files ({total_mb:.1f} MB)")

    ts = time.strftime("%Y%m%d_%H%M")
    archive_base = Path("/workspace") / f"runpod_results_13April2026_{ts}"
    archive_path = shutil.make_archive(
        str(archive_base), "zip",
        root_dir=RESULTS_DIR.parent,
        base_dir=RESULTS_DIR.name)
    zip_mb = Path(archive_path).stat().st_size / 1024**2

    print(f"\n✓ Archive: {archive_path} ({zip_mb:.1f} MB)")
    print(f"\n{'='*60}")
    print("  DOWNLOAD THIS BEFORE STOPPING THE POD!")
    print(f"  RunPod Files → /workspace/ → {Path(archive_path).name}")
    print(f"{'='*60}")

    display(HTML(
        f"<div style='background:#d4edda;padding:12px;border-radius:4px;'>"
        f"<b>Archive ready:</b> <code>{archive_path}</code><br>"
        f"Size: {zip_mb:.1f} MB | Files: {n_files}<br>"
        f"<b>Download before stopping pod!</b></div>"
    ))